# Alignment Bias Eval.

Reproduce results from the paper.

Walkthrough of §3 of the paper. Reads the cached aggregated parquets shipped in `data/`, recomputes every reported number, and rebuilds every figure inline.

Runs on CPU.

Use this if you want to see how the numbers arise rather than just verifying they match.

## 0 · Setup

Installs the package in CPU mode and adds `src/` to `sys.path`. Same Colab-safe pattern as the GPU notebook.

In [ ]:
import importlib, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Environment: {"Colab" if IN_COLAB else "local"}')
print(f'Python:      {sys.version.split()[0]}')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
importlib.invalidate_caches()

import biaseval
print(f'biaseval:    {biaseval.__version__}')

## 1 · Verification gate — all paper numbers

Recomputes every claim in §3 from the cached parquets and asserts each against the value in the paper. If anything ever drifts, the test fails and prints which claim.

In [ ]:
!pytest tests/test_paper_numbers.py -v

## 2 · §3.1 — chat-template-conditional bias

Paired bootstrap of Δ_chat per pair, OLS with HC3 SEs on the pooled effect, and Holm-Bonferroni across the four benchmark tests within each scoring condition. Writes `tables/regression.tex`.

In [ ]:
!python scripts/analysis/chat_template_stats.py

## 3 · §3.2 — BBQ deferral and conditional bias

Splits the BBQ ambiguous-context score into a deferral rate and a conditional bias on the answers a model commits to, for each base–instruct pair.


In [ ]:
!python scripts/analysis/bbq_decomposition.py

## 4 · §3.2 and Appendix A — recoverability framings

Rebound on CrowS-Pairs under each of the six framings (Table 4), and the BBQ decomposition per framing on the shared 6,001-item set (Table 5).


In [ ]:
!python scripts/analysis/framing_stats.py


## 5 · §3.3 — probing + INLP/LEACE

Peak per-layer probe accuracies, direction-cosine analysis (gender + random-pair baseline), and the cross-pair INLP/LEACE effect on gender CrowS-Pairs under the sanity-gated subset.

In [ ]:
!python scripts/analysis/probing_stats.py

## 6 · Figures

Rebuilds each paper figure from the cached parquets. PNG + PDF are written to `figures/`.

In [ ]:
from IPython.display import Image, display

!python scripts/figures/chat_template_dumbbell.py
display(Image('figures/chat_template_dumbbell.png'))

In [ ]:
!python scripts/figures/bbq_quadrant.py
display(Image('figures/bbq_quadrant.png'))

In [ ]:
!python scripts/figures/probing_curves.py
display(Image('figures/probing_curves.png'))

## 7 · Extended views (not in the paper)

Per-pair 8-panel probing grid that backs the Fig 5 claim that the cross-family aggregate isn't an averaging artifact.

In [ ]:
!python scripts/figures/extended_views/probing_8panels.py
display(Image('figures/extended_views/probing_8panels.png'))